# NeuroGolf 2026 : ONNX Submission

Before writing out the submission, we profile each graph on three axes that matter for this competition:

- **Parameters** — every weight/bias tensor contributes.
- **Memory bytes** — raw byte size of initializers.
- **MACs** — multiply-accumulates for `Conv`/`Gemm`/`MatMul` ops on the 30×30 input grid.

The evaluator scores each task with `max(1, 25 - ln(cost))` where `cost = params + memory_bytes + MACs`, so every term pulls its weight. We visualize the distribution, call out the most/least expensive graphs, and summarize the op-type mix across all solved tasks.

## 1. Setup

Per-task ONNX graphs are attached as a Kaggle dataset (one `taskNNN.onnx` per file). Update `DATASET_DIR` if your dataset slug differs.

In [ ]:
import os
import io
import math
import zipfile
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import onnx
import onnx.numpy_helper as onh

# Dataset: one taskNNN.onnx per file.
DATASET_DIR = Path("/kaggle/input/datasets/yash9439/neurogolf-submission")

onnx_files = sorted(DATASET_DIR.glob("task*.onnx"))
if not onnx_files:
    # Fallback: in case the dataset got mounted at a slightly different path.
    onnx_files = sorted(Path("/kaggle/input").rglob("task*.onnx"))

assert onnx_files, f"No task*.onnx files found under {DATASET_DIR}. Attach the dataset."
print(f"Dataset dir: {onnx_files[0].parent}")
print(f"Found {len(onnx_files)} ONNX files.")
print(f"Total size:  {sum(p.stat().st_size for p in onnx_files)/1024:.1f} KB")

OUTPUT_PATH = Path("/kaggle/working/submission.zip")

# Competition constants.
C, H, W = 10, 30, 30
NUM_TASKS = 400

## 2. Load ONNX graphs from disk

We keep every graph in memory so we can both profile it *and* write it into the output zip without re-reading from disk.

In [ ]:
graphs = {}  # task_name -> raw ONNX bytes

for p in onnx_files:
    graphs[p.name] = p.read_bytes()

print(f"Loaded {len(graphs)} ONNX graphs (out of {NUM_TASKS} tasks).")
print(f"Coverage: {len(graphs)/NUM_TASKS:.1%}")
print("Sample task IDs:", sorted(graphs)[:5], "...", sorted(graphs)[-3:])

## 3. Per-graph cost profiler

Exact formula used by the evaluator (public spec):

```
cost  = params + memory_bytes + MACs
score = max(1, 25 - ln(cost))
```

`MACs` counts `Conv` and `Gemm`/`MatMul` only; elementwise ops (`Add`, `Mul`, `Relu`, `Gather`, `Slice`, `Transpose`, `Tile`, `Pad`, `Resize`, `Reshape`) don't contribute to the MAC term even though they drive runtime.

In [ ]:
def profile_onnx(raw_bytes):
    model = onnx.load_model_from_string(raw_bytes)

    tensors = {}
    params = 0
    nbytes = 0

    for init in model.graph.initializer:
        arr = onh.to_array(init)
        tensors[init.name] = arr
        params += arr.size
        nbytes += arr.nbytes

    for node in model.graph.node:
        if node.op_type == "Constant":
            for attr in node.attribute:
                if attr.t:
                    try:
                        arr = onh.to_array(attr.t)
                        if node.output:
                            tensors[node.output[0]] = arr
                        params += arr.size
                        nbytes += arr.nbytes
                    except Exception:
                        pass

    macs = 0
    op_counter = Counter()
    for node in model.graph.node:
        op_counter[node.op_type] += 1
        if node.op_type == "Conv" and len(node.input) >= 2 and node.input[1] in tensors:
            w = tensors[node.input[1]]
            if w.ndim == 4:
                c_out, c_in, kh, kw = w.shape
                macs += c_out * c_in * kh * kw * H * W
        elif node.op_type in ("Gemm", "MatMul") and len(node.input) >= 2 and node.input[1] in tensors:
            w = tensors[node.input[1]]
            if w.ndim == 2:
                macs += w.shape[0] * w.shape[1]

    cost = params + nbytes + macs
    score = max(1.0, 25.0 - math.log(max(cost, 1)))
    return {
        "params": params,
        "bytes": nbytes,
        "macs": macs,
        "cost": cost,
        "score": score,
        "n_nodes": len(model.graph.node),
        "ops": dict(op_counter),
    }


rows = []
op_totals = Counter()
for name in sorted(graphs):
    info = profile_onnx(graphs[name])
    op_totals.update(info["ops"])
    rows.append({
        "task": name.replace(".onnx", ""),
        "params": info["params"],
        "bytes": info["bytes"],
        "macs": info["macs"],
        "cost": info["cost"],
        "score": info["score"],
        "n_nodes": info["n_nodes"],
    })

df = pd.DataFrame(rows).sort_values("cost").reset_index(drop=True)
df.head(10)

## 4. Headline numbers

In [ ]:
solved = len(df)
unsolved = NUM_TASKS - solved
est_total_score = df["score"].sum() + unsolved * 1.0  # unsolved tasks score 1

print(f"Solved tasks:       {solved} / {NUM_TASKS}")
print(f"Unsolved (score 1): {unsolved}")
print(f"Estimated LB total: {est_total_score:.2f}  (sum over 400 tasks)")
print()
print("Cost breakdown over solved tasks:")
print(df[["params", "bytes", "macs", "cost"]].describe().astype(int))

## 5. Cost distribution

Most graphs should cluster at the low end — tiny `Gather`/`Slice`/`Transpose` compositions that cost only a few hundred units. Anything above ~1e6 is almost certainly a learned `Conv` stack, which is where most budget goes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(np.log10(df["cost"].clip(lower=1)), bins=40, color="#4c72b0", edgecolor="white")
axes[0].set_xlabel("log10(cost)")
axes[0].set_ylabel("# tasks")
axes[0].set_title("Cost distribution (log scale)")

axes[1].hist(df["score"], bins=30, color="#55a868", edgecolor="white")
axes[1].set_xlabel("per-task score")
axes[1].set_ylabel("# tasks")
axes[1].set_title("Score distribution over solved tasks")

plt.tight_layout()
plt.show()

## 6. Op-type frequency

A tally of every ONNX node used across the 400-task bundle. The competition bans `Loop`, `Scan`, `NonZero`, `Unique`, `Script`, and `Function` — we assert that none appear.

In [ ]:
banned = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
used_banned = banned & set(op_totals)
assert not used_banned, f"Banned ops present: {used_banned}"

op_series = pd.Series(op_totals).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, max(3, 0.28 * len(op_series))))
op_series.plot.barh(ax=ax, color="#4c72b0")
ax.set_xlabel("# occurrences across all graphs")
ax.set_title("ONNX op-type frequency")
plt.tight_layout()
plt.show()

print(f"Unique op types used: {len(op_series)}")
print(f"No banned ops present.")

## 7. Cheapest and most expensive graphs

In [ ]:
print("Top 10 cheapest graphs (highest per-task score):")
print(df.nsmallest(10, "cost")[["task", "cost", "score", "n_nodes"]].to_string(index=False))

print("\nTop 10 most expensive graphs (room to improve):")
print(df.nlargest(10, "cost")[["task", "cost", "score", "n_nodes"]].to_string(index=False))

## 8. Write the final submission

Repackage the graphs (already in memory) into `/kaggle/working/submission.zip`.

In [ ]:
with zipfile.ZipFile(OUTPUT_PATH, "w", zipfile.ZIP_DEFLATED) as zf_out:
    for name in sorted(graphs):
        zf_out.writestr(name, graphs[name])

size_kb = OUTPUT_PATH.stat().st_size / 1024
print(f"Wrote {OUTPUT_PATH}  ({size_kb:.1f} KB, {len(graphs)} ONNX files)")

# Sanity: re-open and confirm every entry parses.
with zipfile.ZipFile(OUTPUT_PATH, "r") as zf:
    n_ok = 0
    for info in zf.infolist():
        onnx.load_model_from_string(zf.read(info.filename))
        n_ok += 1
print(f"All {n_ok} ONNX graphs in the final zip parse cleanly.")